### Imports

In [68]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore
import faiss
import os
import json
from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    context_recall,
    context_precision,
    faithfulness,
    answer_similarity,
)
from datasets import Dataset

C:\Users\User\AppData\Local\Temp\ipykernel_16052\3024635661.py:10: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_16052\3024635661.py:10: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_16052\3024635661.py:10: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_16

### RAG Pipeline

In [69]:
reader = SimpleDirectoryReader(input_dir = "./data_sample")
document = reader.load_data()

# Splitting sentences without losing sentence coherence
parser = SentenceSplitter(chunk_size = 512, chunk_overlap = 50)
nodes = parser.get_nodes_from_documents(document)

model = Ollama(model = "llama3.2:3b")
embedding = OllamaEmbedding(model_name="nomic-embed-text")

# Setting up non-volatile Faiss DB
faiss_dir = "./faiss_index"
os.makedirs(faiss_dir, exist_ok=True)
vector_store = FaissVectorStore(
    faiss_index=faiss.IndexFlatIP(768)
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(nodes, storage_context = storage_context, embed_model= embedding)
query_engine = index.as_query_engine(llm = model, embed_model = embedding, similarity_top_k = 5)

#### Questions, Contexts & Ground truth

In [70]:
questions = [
    "What is the optimal tilt angle for a site at 40 degrees north latitude?",
    "How often should solar panels be cleaned?",
    "What is the cycle life of LiFePO4 cells at 80% depth of discharge?",
    "What are the benefits of using LiFePO4 batteries in residential storage systems?",
    "What temperature range can lithium iron phosphate batteries operate in?",
    "What type of battery has the lowest initial cost but highest lifetime cost?",
    "What is the energy density of NMC batteries compared to other chemistries?",
    "How does shading affect solar panel output?",
    "Why are BMS components critical for lithium battery systems?",
    "What is the recommended roof load capacity for residential solar installations?"
]
ground_truths = [
    "The optimal tilt angle equals the latitude of the installation site. For a 40 degrees north latitude, the panel should be tilted at 40 degrees from horizontal.",
    "Solar panels should be cleaned every 2-3 months or after heavy dust storms or bird droppings to maintain efficiency and performance.",
    "LiFePO4 cells typically have a cycle life of 3,000-6,000 cycles at 80% depth of discharge.",
    "LiFePO4 batteries offer excellent thermal stability (non-flammable electrolyte), long cycle life, and high safety for residential storage applications.",
    "Lithium iron phosphate batteries can operate in temperatures from 0 to 45 degrees Celsius.",
    "Lead-acid (AGM/Gel) batteries have the lowest initial cost but higher lifetime costs due to frequent replacement requirements.",
    "NMC batteries have an energy density of 150-220 Wh/kg, which is higher than lead-acid (30-40 Wh/kg) but lower than LiFePO4 (90-160 Wh/kg).",
    "Even partial shading on a single cell can reduce the output of an entire string by up to 80%.",
    "The Battery Management System (BMS) is critical for lithium battery systems as it monitors and controls key parameters including cell voltage, temperature, current, and state of charge for safety and performance.",
    "Residential roofs must support at least 30 kg per square meter dead load plus 1.5 kN per square meter live load in snow regions."
]
contexts = [
    "For fixed installations, the optimal tilt angle equals the latitude. At 40 degrees north latitude, this means a 40-degree tilt is optimal for solar panels.",
    "Solar panel maintenance includes cleaning every 2-3 months or after heavy dust storms or bird droppings to maintain efficiency and performance.",
    "Lithium Iron Phosphate (LiFePO4): Cycle life: 3,000-6,000 cycles at 80% depth of discharge",
    "Lithium Iron Phosphate (LiFePO4): Thermal stability: Excellent, non-flammable electrolyte. Best for: Long-duration residential storage, high safety.",
    "Lithium Iron Phosphate (LiFePO4): Operating temperature: 0 to 45 degrees C",
    "Lead-Acid (AGM / Gel): Cost: Lowest initial, highest lifetime (due to replacement)",
    "Nickel Manganese Cobalt (NMC): Energy density: 150-220 Wh/kg",
    "Even partial shading on a single cell can reduce the output of an entire string by up to 80%. A shading study using a solace sensor or PV*SOL software must be performed.",
    "The BMS is the critical safety and performance component of any lithium battery system. It monitors and controls: Cell voltage, Temperature, Current, State of Charge (SOC) estimation",
    "Residential roofs must support a dead load of at least 30 kg per square meter for the mounting hardware and panels, plus a live load of 1.5 kN per square meter for snow regions."
]

### Result

In [71]:
results = []

for i, question in enumerate(questions):
    print("\n" + "=" * 60)
    print("Q:", question)
    r = query_engine.query(question)
    print("A:", r.response)
    results.append({
        "question": question,
        "answer": str(r.response),
        "ground_truth": ground_truths[i],
        "contexts": [node.get_content() for node in r.source_nodes]
    })


Q: What is the optimal tilt angle for a site at 40 degrees north latitude?
A: For a site at 40 degrees north latitude, the optimal tilt angle equals the latitude of the installation site.

Q: How often should solar panels be cleaned?
A: The panels should be cleaned every quarter, with the cleaning process being gentle to avoid damaging the tempered glass cover.

Q: What is the cycle life of LiFePO4 cells at 80% depth of discharge?
A: Approximately 3,000-3,000 cycles.

Q: What are the benefits of using LiFePO4 batteries in residential storage systems?
A: LiFePO4 batteries offer several benefits in residential storage systems, including long cycle life, excellent thermal stability, and high safety. They are suitable for long-duration residential storage and are considered a safe option due to their non-flammable electrolyte and moderate cost. Additionally, LiFePO4 batteries can provide a reliable source of power during grid outages and can be designed to operate efficiently in a variety

### Evaluation